In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

# ==========================================
# STEP 1: LOAD THE DATASET
# ==========================================
# Make sure 'Task_3_and_4_Loan_Data.csv' is uploaded in Colab
df = pd.read_csv('Task_3_and_4_Loan_Data.csv')

# Define Independent Variables (Features) and Dependent Variable (Target)
features = [
    'credit_lines_outstanding',
    'loan_amt_outstanding',
    'total_debt_outstanding',
    'income',
    'years_employed',
    'fico_score'
]
target = 'default'

X = df[features]
y = df[target]

# Train-Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# STEP 2: TRAIN THE CREDIT RISK MODEL
# ==========================================
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluate Model Performance
y_pred_prob = model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_prob)

print("===== MODEL PERFORMANCE =====")
print(f"ROC-AUC Score: {auc_score:.4f}")
print("=============================\n")

# ==========================================
# STEP 3: EXPECTED LOSS FUNCTION
# ==========================================
def calculate_expected_loss(loan_features, trained_model, recovery_rate=0.10):
    """
    Calculates Probability of Default (PD) and Expected Loss (EL)
    for a borrower given loan properties.

    Expected Loss = Loan Amount * PD * (1 - Recovery Rate)
    """
    # Convert input dict to DataFrame if needed
    if isinstance(loan_features, dict):
        input_df = pd.DataFrame([loan_features])
    else:
        input_df = loan_features

    # 1. Estimate Probability of Default (PD)
    pd_probability = trained_model.predict_proba(input_df[features])[:, 1][0]

    # 2. Extract Outstanding Loan Amount
    loan_amount = input_df['loan_amt_outstanding'].values[0]

    # 3. Calculate Loss Given Default (LGD)
    lgd = 1.0 - recovery_rate

    # 4. Expected Loss Formula
    expected_loss = loan_amount * pd_probability * lgd

    return pd_probability, expected_loss

# ==========================================
# STEP 4: TEST WITH A BORROWER RECORD
# ==========================================
# Example borrower from your dataset:
sample_borrower = {
    'credit_lines_outstanding': 5,
    'loan_amt_outstanding': 1958.93,
    'total_debt_outstanding': 8228.75,
    'income': 26648.44,
    'years_employed': 2,
    'fico_score': 572
}

pd_val, el_val = calculate_expected_loss(sample_borrower, model, recovery_rate=0.10)

print("===== BORROWER VALUATION RESULTS =====")
print(f"Loan Outstanding:        ${sample_borrower['loan_amt_outstanding']:,.2f}")
print(f"Probability of Default:  {pd_val * 100:.2f}%")
print(f"Recovery Rate:          10.0%")
print(f"EXPECTED LOSS:          ${el_val:,.2f}")
print("======================================")

===== MODEL PERFORMANCE =====
ROC-AUC Score: 1.0000

===== BORROWER VALUATION RESULTS =====
Loan Outstanding:        $1,958.93
Probability of Default:  100.00%
Recovery Rate:          10.0%
EXPECTED LOSS:          $1,763.04
